# LocusZoom / summary-statistic export

Turns pipeline or meta-analysis results into the tab-separated, bgzipped
format LocusZoom (and `locuscompare_setaxes.R`) expects, with one row per
variant and consistent `CHROM / POS / REF / ALT` columns.

Three cases are covered:

1. a single-ancestry GWAS written by the pipeline,
2. a PLINK `.meta` meta-analysis,
3. a METAL `.TBL` meta-analysis, where the effect allele has to be aligned to
   ALT before the file uploaded to LocusZoom.

This reads straight from the bucket rather than through a gcsfuse mount, so it
runs anywhere with credentials.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The shared pipeline config lives in the repo root; point GWAS_SCRIPTS_DIR
# at it if this notebook is run from somewhere else.
SCRIPTS_DIR = Path(os.environ.get('GWAS_SCRIPTS_DIR', Path.cwd().parent))
sys.path.insert(0, str(SCRIPTS_DIR))

import gwas_config as cfg
import gcs_io

pd.options.mode.chained_assignment = None
cfg.describe()

In [ ]:
bucket = gcs_io.get_bucket()
print(bucket.name)

## 1. Single-ancestry GWAS

`study` names one cohort for a stratified analysis (survival results are
written per cohort); leave it `None` for the pooled LR analysis across GP2 cohorts.

In [ ]:
LABEL = 'AAC'          # ancestry, or the ancestry a non-NBA cohort was run as
ANALYSIS = 'CS_disc'   # CS, CS_disc, LT, SURV
MODALITY = 'NBA'       # 'NBA', 'WGS' or 'non_NBA'
COHORT = None          # non-NBA cohort name, when MODALITY == 'non_NBA'
STUDY = None           # study arm for stratified results, else None

model = cfg.MODEL_DIRS[ANALYSIS]
arm = STUDY or 'case_control'
run_dir = f'{LABEL}_{ANALYSIS}_CHROMS_1_22'

if MODALITY == 'non_NBA':
    parts = ['non_NBA', COHORT, ANALYSIS]
elif MODALITY == 'WGS':
    parts = [LABEL, 'WGS', ANALYSIS]
else:
    parts = [LABEL, ANALYSIS]

results_path = '/'.join([cfg.PREFIX, f'R{cfg.RELEASE}_prep', 'progression', *parts,
                         run_dir, 'gwas_results', model,
                         f'{LABEL}_{arm}_y_allresults.tsv'])
print(results_path)

In [ ]:
df = gcs_io.blob_as_csv(bucket, results_path, sep='\t')
print(f'{len(df)} rows')
df = df.dropna(how='any', axis=0)
print(f'{len(df)} rows after dropping incomplete records')

df['CHROM'] = df['#CHROM'].str.replace('chr', '').astype(int)
df = df.drop_duplicates(subset=['CHROM', 'POS', 'ID'])
display(df.CHROM.value_counts().sort_index())

all_chroms = df.sort_values(['CHROM', 'POS'])
display(all_chroms.head())

In [ ]:
# Sanity checks before exporting.
# A1 should be the ALT allele
# LongGWAS should have made A1 the minor allele
print('A1 != ALT:', (all_chroms.ALT != all_chroms.A1).sum())
print('A1_FREQ > 0.5:', (all_chroms.A1_FREQ > 0.5).sum())
print('A1_FREQ < 0.01:', (all_chroms.A1_FREQ < 0.01).sum())

In [ ]:
# Keep A1 and OBS_CT: downstream tools need the effect allele and the sample
# size. The test-statistic column differs by model.
STAT_COL = {'CS_disc': 'Z_STAT', 'CS': 'T_STAT', 'SURV': None, 'LT': None}[ANALYSIS]

cols = ['CHROM', 'POS', 'REF', 'ALT', 'A1', 'OBS_CT', 'A1_FREQ', 'P', 'BETA', 'SE']
if ANALYSIS == 'LT':
    # the longitudinal model reports slope and intercept separately
    cols = ['CHROM', 'POS', 'REF', 'ALT', 'A1', 'OBS_CT', 'A1_FREQ',
            'Ps', 'BETAs', 'SEs', 'Pi', 'BETAi', 'SEi']
elif STAT_COL:
    cols.append(STAT_COL)

export = all_chroms[cols].rename(columns={'A1_FREQ': 'allele_freq'})
display(export.head())

In [ ]:
out_dir = Path('locuszoom') / f'R{cfg.RELEASE}' / MODALITY
out_dir.mkdir(parents=True, exist_ok=True)

suffix = f'_{STUDY}' if STUDY else ''
out_path = out_dir / f'{LABEL}_{ANALYSIS}_R{cfg.RELEASE}{suffix}_all_chrom_results.tsv.gz'
export.to_csv(out_path, sep='\t', index=False, compression='gzip')
print(f'wrote {out_path}')

## 2. PLINK meta-analysis

In [ ]:
ANALYSIS = 'CS_disc'
META_LABEL = 'EUR_AJ_with_EUR_WGS'

results_path = '/'.join([cfg.PREFIX, f'R{cfg.RELEASE}_prep', 'progression', 'meta',
                         'PLINK', ANALYSIS, f'{ANALYSIS}_{META_LABEL}.meta'])

plink_meta = gcs_io.blob_as_csv(bucket, results_path, sep=r'\s+').dropna(how='any')
print(f'{len(plink_meta)} variants')
display(plink_meta.head())

## 3. METAL meta-analysis

METAL reports its own effect allele (`Allele1`), which is whichever allele it
saw first, so it is not always ALT. Rows where it is REF are flipped - alleles
swapped, frequency complemented, effect negated - so that the exported file
always has the ALT allele as the effect allele. SE and P are unchanged by a
flip.

In [ ]:
ANALYSIS = 'CS_disc'
METAL_LABEL = 'EUR_AJ_with_EUR_WGS'
EFFECTS = 'random'     # 'fixed' uses Effect/StdErr/Pvalue, 'random' the ARE columns

results_path = '/'.join([cfg.PREFIX, f'R{cfg.RELEASE}_prep', 'progression', 'meta',
                         'METAL', ANALYSIS, f'{ANALYSIS}_{METAL_LABEL}.TBL'])

df = gcs_io.blob_as_csv(bucket, results_path, sep=r'\s+')
print(f'{len(df)} variants')
df = df.dropna(how='any', axis=0)
print(f'{len(df)} after dropping incomplete records')

ids = df['MarkerName'].str.split(':')
df['CHROM'] = ids.str[0].str.replace('chr', '').astype(int)
df['POS'] = ids.str[1].astype(int)
df['REF'] = ids.str[2]
df['ALT'] = ids.str[3]
df = df.rename(columns={'Pvalue': 'P', 'Effect': 'BETA', 'StdErr': 'SE'})

if EFFECTS == 'fixed':
    effect_cols = ['BETA', 'SE', 'P']
else:
    effect_cols = ['EffectARE', 'StdErrARE', 'PvalueARE']
df = df[['CHROM', 'POS', 'REF', 'ALT', 'Allele1', 'Allele2', 'Freq1'] + effect_cols]

In [ ]:
# METAL writes alleles in lower case, so compare on upper case.
a1 = df['Allele1'].str.upper()
a2 = df['Allele2'].str.upper()
flip = (a1 == df['REF'].str.upper()) & (a2 == df['ALT'].str.upper())
print(f'flipping {flip.sum()} of {len(df)} variants so the effect allele is ALT')

df.loc[flip, ['Allele1', 'Allele2']] = df.loc[flip, ['Allele2', 'Allele1']].values
df.loc[flip, 'Freq1'] = 1 - df.loc[flip, 'Freq1']
df.loc[flip, effect_cols[0]] = -df.loc[flip, effect_cols[0]]

df = df.sort_values(['CHROM', 'POS']).reset_index(drop=True)
display(df.head())

In [ ]:
out_dir = Path('locuszoom') / f'R{cfg.RELEASE}' / 'METAL' / EFFECTS
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f'{ANALYSIS}_{METAL_LABEL}_METAL.tsv.gz'
df.to_csv(out_path, sep='\t', index=False, compression='gzip')
print(f'wrote {out_path}')